# 02 テキスト前処理

質問タイトルに対してクリーニング・トークナイズ・ストップワード除去を適用し、
後続のテキストマイニング（Day 7）に備えたパイプラインを構築する。

> **Note**: 質問本文（body）は Day 7 のミクロ分析時に `include_body=True` で別途取得する。
> 本ノートブックはタイトルを使って前処理パイプラインの動作を確認する。


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from src.preprocessing.cleaner import Cleaner
from src.preprocessing.tokenizer import Tokenizer

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (12, 5)

df_raw = pd.read_parquet('../data/processed/questions.parquet')
print(f"読み込み: {len(df_raw):,}件")
print(df_raw[['title', 'language', 'year']].head(5).to_string())

## 1. クリーニング（Cleaner）

In [ ]:
cleaner = Cleaner()
df = cleaner.clean_questions(df_raw)

# 変換前後の比較
print(f"クリーニング前: {len(df_raw):,}件")
print(f"クリーニング後: {len(df):,}件")
print()
print("creation_date の型:", df['creation_date'].dtype)
print("サンプル:")
print(df[['question_id', 'title', 'creation_date', 'language', 'year']].head(3).to_string())

## 2. トークナイズ（Tokenizer）

In [ ]:
tokenizer = Tokenizer()

# Python 質問のタイトルでトークナイズの動作を確認
python_titles = df[df['language'] == 'python']['title'].head(8)

for title in python_titles:
    tokens = tokenizer.tokenize(title)
    print(f"原文  : {title}")
    print(f"トークン: {tokens[:12]}")
    print()

## 3. ストップワード除去の効果

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

sample_title = "How do I get a list of all the values in a Python dictionary?"
raw_tokens = [t.lower() for t in word_tokenize(sample_title) if t.isalpha()]
clean_tokens = tokenizer.tokenize(sample_title)

print(f"原文   : {sample_title}")
print(f"全トークン ({len(raw_tokens)}): {raw_tokens}")
print(f"除去後  ({len(clean_tokens)}): {clean_tokens}")
print(f"除去された語: {set(raw_tokens) - set(clean_tokens)}")

## 4. 言語別 頻出語ランキング

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

for ax, lang in zip(axes.flatten(), ['python', 'javascript', 'java', 'go']):
    titles = df[df['language'] == lang]['title']
    all_tokens = []
    for t in titles:
        all_tokens.extend(tokenizer.tokenize(t))

    counter = Counter(all_tokens)
    top = counter.most_common(15)
    words, counts = zip(*top)

    ax.barh(list(words)[::-1], list(counts)[::-1], color='#4f8ef7', alpha=0.8)
    ax.set_title(f'{lang.capitalize()} — タイトル頻出語 Top15')
    ax.set_xlabel('出現回数')

plt.tight_layout()
plt.savefig('../data/analysis/fig_top_words.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. 前処理済みデータの保存

In [ ]:
# クリーニング済みデータを questions.parquet に上書き保存
out_path = '../data/processed/questions.parquet'
df.to_parquet(out_path, index=False)
print(f"保存完了: {out_path}")
print(f"件数: {len(df):,}件 / 列数: {len(df.columns)}")
print(f"列: {list(df.columns)}")

## 前処理まとめ

| ステップ | 処理内容 |
|---|---|
| Cleaner | タイトル欠損除去・日時型変換・重複除去 |
| Tokenizer | 小文字化・NLTK word_tokenize・英数字のみ抽出 |
| Stopwords | 英語ストップワード + ドメイン固有語（get/use/set 等）除去 |

本文（body）は Day 7 のミクロ分析で `include_body=True` を使い別途取得する。
そこでは HTML タグ除去（`Cleaner.clean_body()`）も適用する。
